# Phase 0 + 1 — ยืนยันสเปกโมเดล + วัด Baseline

Notebook นี้ทำ 2 อย่างของ pipeline ([แผนเต็ม](../llm_pruning_project_plan.md)):
- **Phase 0** — โหลด Typhoon 2 3B แล้วยืนยัน `config.json` จริง + นับพารามิเตอร์จริง (ห้ามเชื่อตัวเลขสมมติในแผนจนกว่าจะยืนยัน)
- **Phase 1** — วัด baseline 3 แกนของโมเดล **เดิม** ไว้เป็นจุดอ้างอิง (B1) สำหรับเทียบทุก phase: คุณภาพในโดเมน / นอกโดเมน / ประสิทธิภาพ

## วิธีรันบน Kaggle
1. Settings → Accelerator = **GPU T4 x2** หรือ **P100** (16GB พอสำหรับ 3B ที่ BF16)
2. Settings → Internet = **On** (ต้องโหลดโมเดลจาก Hugging Face)
3. ใส่ HF token ใน **Add-ons → Secrets** ชื่อ `HF_TOKEN` (ถ้าโมเดล gated)
4. รันบนลงล่าง • artifact (ผล baseline) ถูกเซฟไว้ที่ `/kaggle/working/` แล้ว Save Version เพื่อเก็บเป็น output

> ✅ Checklist ที่ปิดได้หลังรัน notebook นี้: Phase 0 ทั้งหมด + Phase 1 (setup, latency, RAM, size, out-of-domain). ส่วน **in-domain** ต้องรอ test set จาก Phase 2

import os
# --- ให้โมเดล/แคชของ HuggingFace ลงไดรฟ์ D (ต้องตั้งก่อน import transformers) ---
# โมเดล 3B ~6.4GB ถ้าไม่ตั้ง จะไปลง C:\Users\...\.cache ซึ่งเหลือน้อย
if os.path.isdir(r"D:\Code\pholama"):
    os.environ.setdefault("HF_HOME", r"D:\Code\pholama\.cache\hf")
    os.environ.setdefault("HF_HUB_DISABLE_SYMLINKS_WARNING", "1")

import json, time, gc, platform
import torch
from transformers import AutoConfig, AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

# ===== ตั้งค่าโปรเจกต์ =====
MODEL_ID = "scb10x/llama3.2-typhoon2-3b-instruct"   # ยืนยัน id จริงของ Typhoon 2 3B
DTYPE = torch.bfloat16
# RTX 3050 Laptop = 4GB VRAM → ต้องโหลด 4-bit (BF16 6.4GB จะ OOM แน่นอน)
# ตั้ง False เมื่อรันบน GPU ใหญ่/Kaggle 16GB เพื่อวัด baseline BF16 ตรงๆ
LOAD_4BIT = True
# OUT_DIR: บน Kaggle ใช้ /kaggle/working, บน local ใช้ราก project (ไม่ใช่ notebooks/)
if os.path.isdir("/kaggle/working"):
    OUT_DIR = "/kaggle/working"
elif os.path.isdir(r"D:\Code\pholama"):
    OUT_DIR = r"D:\Code\pholama"
else:
    OUT_DIR = ".."

# HF token (Kaggle Secrets หรือ env var HF_TOKEN)
try:
    from kaggle_secrets import UserSecretsClient
    os.environ.setdefault("HF_TOKEN", UserSecretsClient().get_secret("HF_TOKEN"))
except Exception:
    pass

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device, "| 4-bit:", LOAD_4BIT, "| HF_HOME:", os.environ.get("HF_HOME", "(default)"))
if device == "cuda":
    p = torch.cuda.get_device_properties(0)
    print(f"GPU: {p.name} | VRAM: {p.total_memory/1e9:.2f} GB")
print("torch:", torch.__version__)

In [1]:
# บน Kaggle: uncomment เพื่อติดตั้ง deps (Kaggle มี torch/transformers อยู่แล้ว)
# %pip install -q -U "transformers>=4.46" accelerate bitsandbytes datasets sentencepiece
# %pip install -q lm-eval   # เปิดเมื่อจะรัน harness เต็ม
#
# บน local (env บน D): deps ครบแล้วผ่าน requirements.txt — ไม่ต้องติดตั้งซ้ำ
print("deps: ใช้ env ที่ตั้งไว้ (local) หรือ uncomment %pip ด้านบน (Kaggle)")

deps: ใช้ env ที่ตั้งไว้ (local) หรือ uncomment %pip ด้านบน (Kaggle)


In [2]:
import os
# --- ให้โมเดล/แคชของ HuggingFace ลงไดรฟ์ D (ต้องตั้งก่อน import transformers) ---
# โมเดล 3B ~6.4GB ถ้าไม่ตั้ง จะไปลง C:\Users\...\.cache ซึ่งเหลือน้อย
if os.path.isdir(r"D:\Code\pholama"):
    os.environ.setdefault("HF_HOME", r"D:\Code\pholama\.cache\hf")

import json, time, gc, platform
import torch
from transformers import AutoConfig, AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

# ===== ตั้งค่าโปรเจกต์ =====
MODEL_ID = "scb10x/llama3.2-typhoon2-3b-instruct"   # ยืนยัน id จริงของ Typhoon 2 3B
DTYPE = torch.bfloat16
# RTX 3050 Laptop = 4GB VRAM → ต้องโหลด 4-bit (BF16 6.4GB จะ OOM แน่นอน)
# ตั้ง False เมื่อรันบน GPU ใหญ่/Kaggle 16GB เพื่อวัด baseline BF16 ตรงๆ
LOAD_4BIT = True
OUT_DIR = "/kaggle/working" if os.path.isdir("/kaggle/working") else "."

# HF token (Kaggle Secrets หรือ env var HF_TOKEN)
try:
    from kaggle_secrets import UserSecretsClient
    os.environ.setdefault("HF_TOKEN", UserSecretsClient().get_secret("HF_TOKEN"))
except Exception:
    pass

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device, "| 4-bit:", LOAD_4BIT, "| HF_HOME:", os.environ.get("HF_HOME", "(default)"))
if device == "cuda":
    p = torch.cuda.get_device_properties(0)
    print(f"GPU: {p.name} | VRAM: {p.total_memory/1e9:.2f} GB")
print("torch:", torch.__version__)

device: cuda | 4-bit: True | HF_HOME: D:\Code\pholama\.cache\hf
GPU: NVIDIA GeForce RTX 3050 Laptop GPU | VRAM: 4.29 GB
torch: 2.6.0+cu124


## Phase 0 — ยืนยันสเปกโมเดลฐาน

โหลดแค่ `config.json` ก่อน (เบา ไม่ต้องโหลด weight) แล้วเทียบกับตัวเลขที่แผนสมมติไว้ (Llama 3.2 3B)

In [3]:
cfg = AutoConfig.from_pretrained(MODEL_ID, token=os.environ.get("HF_TOKEN"))

# ค่าที่แผนสมมติไว้ — ใช้ตรวจว่าตรงจริงไหม
PLAN_ASSUMED = {
    "hidden_size": 3072,
    "num_hidden_layers": 28,
    "num_attention_heads": 24,
    "num_key_value_heads": 8,
    "head_dim": 128,
    "intermediate_size": 8192,
    "vocab_size": 128256,
    "tie_word_embeddings": True,
}

print(f"{'key':<24}{'จริง':<14}{'แผนสมมติ':<14}{'ตรง?'}")
print("-" * 60)
for k, assumed in PLAN_ASSUMED.items():
    actual = getattr(cfg, k, "(ไม่มี)")
    ok = "✅" if actual == assumed else "❌ ต่าง!"
    print(f"{k:<24}{str(actual):<14}{str(assumed):<14}{ok}")

print("\n⚠️ ถ้ามี ❌ ให้ไปแก้ตาราง param/RAM ใน llm_pruning_project_plan.md ให้ตรงค่าจริง")

key                     จริง          แผนสมมติ      ตรง?
------------------------------------------------------------
hidden_size             3072          3072          ✅
num_hidden_layers       28            28            ✅
num_attention_heads     24            24            ✅
num_key_value_heads     8             8             ✅
head_dim                128           128           ✅
intermediate_size       8192          8192          ✅
vocab_size              128256        128256        ✅
tie_word_embeddings     True          True          ✅

⚠️ ถ้ามี ❌ ให้ไปแก้ตาราง param/RAM ใน llm_pruning_project_plan.md ให้ตรงค่าจริง


In [4]:
# โหลดโมเดล — 4-bit สำหรับ VRAM น้อย (4GB) | BF16 สำหรับ GPU ใหญ่
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, token=os.environ.get("HF_TOKEN"))
load_kwargs = dict(token=os.environ.get("HF_TOKEN"))
if LOAD_4BIT:
    load_kwargs["quantization_config"] = BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_use_double_quant=True)
    load_kwargs["device_map"] = "auto"
else:
    load_kwargs["torch_dtype"] = DTYPE
    load_kwargs["device_map"] = device
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, **load_kwargs)
model.eval()
print("โหลดโมเดลสำเร็จ |", "4-bit (nf4)" if LOAD_4BIT else str(DTYPE))

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

โหลดโมเดลสำเร็จ | 4-bit (nf4)


In [5]:
# นับพารามิเตอร์ — คำนวณจาก config (เชื่อถือได้เสมอ)
# ⚠️ ห้ามใช้ sum(p.numel()) ตอนโหลด 4-bit: bitsandbytes pack 2 ค่า/byte → numel() ได้ครึ่งเดียว
def count_params_from_config(cfg):
    h = cfg.hidden_size
    n_layers = cfg.num_hidden_layers
    kv = cfg.num_key_value_heads
    hd = getattr(cfg, "head_dim", h // cfg.num_attention_heads)
    inter = cfg.intermediate_size
    vocab = cfg.vocab_size
    q = h * (cfg.num_attention_heads * hd)
    k = h * (kv * hd)
    v = h * (kv * hd)
    o = (cfg.num_attention_heads * hd) * h
    attn = q + k + v + o
    mlp = 3 * h * inter                 # gate + up + down
    norms = 2 * h                       # input_layernorm + post_attention_layernorm
    per_layer = attn + mlp + norms
    transformer = per_layer * n_layers + h    # + final norm
    embedding = vocab * h               # tied → นับครั้งเดียว
    if not getattr(cfg, "tie_word_embeddings", True):
        embedding *= 2                  # input + output แยกกัน
    return {"total": transformer + embedding, "embedding": embedding,
            "transformer": transformer, "per_layer": per_layer}

pc = count_params_from_config(cfg)
total = pc["total"]; embedding_total = pc["embedding"]
transformer_params = pc["transformer"]; n_layers = cfg.num_hidden_layers

def b(x):
    return f"{x/1e9:.3f}B ({x/1e6:.1f}M)"

print("== คำนวณจาก config ==")
print(f"รวมทั้งหมด          : {b(total)}")
print(f"embedding (tied={getattr(cfg,'tie_word_embeddings',True)}): {b(embedding_total)}  = {embedding_total/total*100:.1f}%")
print(f"transformer layers   : {b(transformer_params)}  = {transformer_params/total*100:.1f}%")
print(f"ต่อ 1 layer          : {b(pc['per_layer'])}")
print("\nเทียบแผน: รวม ~3.21B, embedding ~12.3%, transformer ~87.7%")

# footprint จริงบน VRAM (4-bit pack) ไว้ดูประกอบ
try:
    print(f"\nmemory footprint จริง (4-bit): {model.get_memory_footprint()/1e9:.2f} GB")
except Exception:
    pass

== คำนวณจาก config ==
รวมทั้งหมด          : 3.213B (3212.7M)
embedding (tied=True): 0.394B (394.0M)  = 12.3%
transformer layers   : 2.819B (2818.7M)  = 87.7%
ต่อ 1 layer          : 0.101B (100.7M)

เทียบแผน: รวม ~3.21B, embedding ~12.3%, transformer ~87.7%

memory footprint จริง (4-bit): 2.20 GB


## Phase 1 — Baseline แกนที่ 3: ประสิทธิภาพ (ขนาด / RAM / latency)

In [6]:
# ขนาด weight ตามจำนวน param × bytes/param
# หมายเหตุ: ถ้า LOAD_4BIT ค่า weight_gb นี้คือ "BF16 เชิงทฤษฎี" (ไว้เทียบ) ไม่ใช่ที่ใช้จริงบน VRAM
bytes_per_param = torch.finfo(DTYPE).bits // 8
weight_gb = total * bytes_per_param / 1e9
print(f"ขนาด weight (BF16 ทฤษฎี): {weight_gb:.2f} GB  ({bytes_per_param} bytes/param)")
if LOAD_4BIT:
    print(f"ขนาด weight (4-bit จริง โดยประมาณ): ~{total*0.55/1e9:.2f} GB")

# RAM/VRAM จริงที่ใช้ตอนนี้
if device == "cuda":
    torch.cuda.synchronize()
    print(f"VRAM allocated      : {torch.cuda.memory_allocated()/1e9:.2f} GB")
    print(f"VRAM reserved (peak): {torch.cuda.max_memory_reserved()/1e9:.2f} GB")

ขนาด weight (BF16 ทฤษฎี): 6.43 GB  (2 bytes/param)
ขนาด weight (4-bit จริง โดยประมาณ): ~1.77 GB
VRAM allocated      : 2.24 GB
VRAM reserved (peak): 2.29 GB


In [7]:
# Latency / throughput — generate แล้ววัด tokens/วินาที (เฉลี่ยหลายรอบ)
@torch.no_grad()
def benchmark_generation(prompt, max_new_tokens=128, n_runs=3, warmup=1):
    msgs = [{"role": "user", "content": prompt}]
    inputs = tokenizer.apply_chat_template(
        msgs, add_generation_prompt=True, return_tensors="pt", return_dict=True
    ).to(model.device)
    for _ in range(warmup):
        model.generate(**inputs, max_new_tokens=16, do_sample=False)
    times = []
    for _ in range(n_runs):
        if device == "cuda":
            torch.cuda.synchronize()
        t0 = time.perf_counter()
        out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
        if device == "cuda":
            torch.cuda.synchronize()
        times.append(time.perf_counter() - t0)
    gen_tokens = out.shape[1] - inputs["input_ids"].shape[1]
    avg = sum(times) / len(times)
    return {"avg_sec": avg, "gen_tokens": gen_tokens, "tok_per_sec": gen_tokens / avg}

perf = benchmark_generation("อธิบายขั้นตอนการยื่นขอทุนการศึกษาโดยย่อ")
print(f"latency: {perf['avg_sec']:.2f}s / {perf['gen_tokens']} tokens  →  {perf['tok_per_sec']:.1f} tok/s")

[transformers] Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


[transformers] Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


[transformers] Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


[transformers] Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


latency: 12.07s / 128 tokens  →  10.6 tok/s


## Phase 1 — Baseline แกนที่ 2: ความสามารถนอกโดเมน (โค้ด/คณิต)

เก็บผลตรงนี้ไว้ **พิสูจน์การถดถอย** ภายหลัง — หลัง specialize ค่าพวกนี้ *ควรตก* (เป็นเป้าหมาย ไม่ใช่บั๊ก) ตอนนี้แค่ probe เชิงคุณภาพ; การวัดเป็นตัวเลขจริงให้ใช้ `lm-eval-harness` (เช่น `gsm8k`, `humaneval`) เมื่อพร้อม

In [8]:
@torch.no_grad()
def chat(prompt, max_new_tokens=256):
    msgs = [{"role": "user", "content": prompt}]
    inputs = tokenizer.apply_chat_template(
        msgs, add_generation_prompt=True, return_tensors="pt", return_dict=True
    ).to(model.device)
    out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    return tokenizer.decode(out[0, inputs["input_ids"].shape[1]:], skip_special_tokens=True)

ood_probes = [
    "Write a Python function that returns the nth Fibonacci number.",
    "ร้านขายส้ม 3 กิโล กิโลละ 25 บาท ทอนจากแบงค์ร้อยเท่าไร?",
]
for p in ood_probes:
    print("Q:", p)
    print("A:", chat(p, max_new_tokens=200)[:500])
    print("-" * 60)

[transformers] Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Q: Write a Python function that returns the nth Fibonacci number.


[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


[transformers] Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


A: Here's a Python function that returns the nth Fibonacci number:

```python
def fibonacci(n):
    if n <= 0:
        return "Input should be positive integer"
    elif n == 1:
        return 0
    elif n == 2:
        return 1
    else:
        a, b = 0, 1
        for _ in range(2, n + 1):
            a, b = b, a + b
        return b
```

This function uses a loop to calculate the nth Fibonacci number. It handles the base cases (n = 1 and n = 2) and returns an error message for non-positive integ
------------------------------------------------------------
Q: ร้านขายส้ม 3 กิโล กิโลละ 25 บาท ทอนจากแบงค์ร้อยเท่าไร?


A: ร้านขายส้ม 3 กิโลกรัม กิโลกรัมละ 25 บาท ดังนั้นราคาส้มทั้งหมดจะเป็น 3 x 25 = 75 บาท

แบงค์ร้อยมีค่าเท่ากับ 1000 บาท ดังนั้นเมื่อทอนจากแบงค์ร้อย จะได้เงินทอนเป็น 1000 - 75 = 925 บาท

ดังนั้นเงินทอนที่ได้จากแบงค์ร้อย 1000 บาท จะเป็น 925 บาท
------------------------------------------------------------


## Phase 1 — Baseline แกนที่ 1: คุณภาพในโดเมน

⏳ **ต้องรอ test set จาก Phase 2** (`02_dataset.ipynb`) ตอนนี้ทำได้แค่ probe เชิงคุณภาพ เมื่อมี test set แล้วให้ attach เป็น Kaggle Dataset input แล้วรัน cell ด้านล่าง

In [9]:
# probe ชั่วคราว (เปลี่ยนเป็น loop บน test set จริงเมื่อมี Phase 2)
for p in ["หอพักนักศึกษาเปิดให้ลงทะเบียนช่วงไหน?", "What documents are required to apply for a scholarship?"]:
    print("Q:", p)
    print("A:", chat(p, max_new_tokens=200)[:500])
    print("-" * 60)

# --- TEMPLATE สำหรับเมื่อมี test set จริง ---
# from datasets import load_dataset
# test = load_dataset("json", data_files="/kaggle/input/<dataset>/test.jsonl")["train"]
# preds = [chat(ex["question"]) for ex in test]
# คำนวณ metric (เช่น LLM-as-judge / ROUGE / exact-match ตามรูปแบบคำตอบ)

[transformers] Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Q: หอพักนักศึกษาเปิดให้ลงทะเบียนช่วงไหน?


[transformers] Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


A: หอพักนักศึกษาเปิดให้ลงทะเบียนในช่วงเปิดภาคการศึกษาใหม่ โดยทั่วไปจะเริ่มตั้งแต่เดือนสิงหาคมถึงกันยายนของทุกปี ขึ้นอยู่กับมหาวิทยาลัยและประเภทของหอพักที่มีให้บริการ ควรตรวจสอบข้อมูลจากมหาวิทยาลัยหรือเว็บไซต์ของหอพักนั้นๆ เพื่อข้อมูลที่ถูกต้องและเป็นปัจจุบัน
------------------------------------------------------------
Q: What documents are required to apply for a scholarship?


A: To apply for a scholarship, you typically need to provide the following documents:

1. **Application Form**: A completed application form that includes your personal information, academic background, and contact details.

2. **Transcripts**: Official transcripts of your academic records, showing your grades and completion status.

3. **Letters of Recommendation**: Letters from professors, employers, or other individuals who can vouch for your academic or professional abilities.

4. **Resume/CV**
------------------------------------------------------------


## บันทึกผล baseline (B1) เป็น artifact

เซฟเป็น JSON ที่ `/kaggle/working/` แล้ว **Save Version** เพื่อเก็บเป็น output ของ notebook (ใช้เทียบกับ B2/B3 ใน Phase 7)

In [10]:
baseline = {
    "label": "B1_original_typhoon2_3b",
    "model_id": MODEL_ID,
    "dtype": str(DTYPE),
    "load_4bit": LOAD_4BIT,
    "config": {k: getattr(cfg, k, None) for k in PLAN_ASSUMED},
    "params": {
        "total": total,
        "embedding": embedding_total,
        "transformer": transformer_params,
        "per_layer_avg": transformer_params // n_layers,
    },
    "efficiency": {
        "weight_gb_bf16_theoretical": round(weight_gb, 3),
        "vram_allocated_gb": round(torch.cuda.memory_allocated()/1e9, 3) if device == "cuda" else None,
        "tok_per_sec": round(perf["tok_per_sec"], 2),
        "gen_latency_sec": round(perf["avg_sec"], 3),
    },
    "in_domain": "TODO: รอ test set Phase 2",
    "out_of_domain": "TODO: รัน lm-eval (gsm8k/humaneval) เพื่อเก็บตัวเลข",
    "env": {"platform": platform.platform(), "torch": torch.__version__},
}

path = os.path.join(OUT_DIR, "baseline_B1.json")
with open(path, "w", encoding="utf-8") as f:
    json.dump(baseline, f, ensure_ascii=False, indent=2)
print("เซฟแล้ว:", path)
print(json.dumps(baseline, ensure_ascii=False, indent=2))

เซฟแล้ว: .\baseline_B1.json
{
  "label": "B1_original_typhoon2_3b",
  "model_id": "scb10x/llama3.2-typhoon2-3b-instruct",
  "dtype": "torch.bfloat16",
  "load_4bit": true,
  "config": {
    "hidden_size": 3072,
    "num_hidden_layers": 28,
    "num_attention_heads": 24,
    "num_key_value_heads": 8,
    "head_dim": 128,
    "intermediate_size": 8192,
    "vocab_size": 128256,
    "tie_word_embeddings": true
  },
  "params": {
    "total": 3212749824,
    "embedding": 394002432,
    "transformer": 2818747392,
    "per_layer_avg": 100669549
  },
  "efficiency": {
    "weight_gb_bf16_theoretical": 6.425,
    "vram_allocated_gb": 2.251,
    "tok_per_sec": 10.61,
    "gen_latency_sec": 12.065
  },
  "in_domain": "TODO: รอ test set Phase 2",
  "out_of_domain": "TODO: รัน lm-eval (gsm8k/humaneval) เพื่อเก็บตัวเลข",
  "env": {
    "platform": "Windows-10-10.0.26200-SP0",
    "torch": "2.6.0+cu124"
  }
}


## ✅ สรุปสิ่งที่ปิด checklist ได้ + ขั้นต่อไป

**ปิดได้หลังรัน notebook นี้** (ไปติ๊กใน [`PROGRESS.md`](../PROGRESS.md)):
- Phase 0 — ยืนยัน config + นับ param จริง
- Phase 1 — setup, latency, RAM, ขนาดไฟล์, out-of-domain (probe)

**ยังค้าง:**
- in-domain quality → ทำใน Phase 2 (มี test set ก่อน)
- ตัวเลข out-of-domain เป็นทางการ → รัน `lm-eval-harness`

**ขั้นต่อไป:** `02_dataset.ipynb` — สร้างชุด Q&A กิจการนักศึกษา (เก็บ `test.jsonl` ไว้ย้อนมาวัด in-domain ของ B1 ที่นี่)